In [6]:
import pandas as pd

# Load the 5 main datasets
orders = pd.read_csv('olist_orders_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

print("Files loaded successfully")

Files loaded successfully


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Explore the reviews dataset
print("Shape (rows, columns):", reviews.shape)
print()
print("First rows:")
reviews.head()

Shape (rows, columns): (99224, 7)

First rows:


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
# Check for missing values in the reviews dataset
print("Missing values per column:")
print(reviews.isnull().sum())
print()
print("Duplicate rows:", reviews.duplicated().sum())

Missing values per column:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Duplicate rows: 0


In [ ]:
# Fill empty comment fields instead of dropping rows (the review_score is what matters)
reviews['review_comment_title'] = reviews['review_comment_title'].fillna('No comment')
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('No comment')

# Confirm there are no more missing values
print("Missing values after cleaning:")
print(reviews.isnull().sum())

Missing values after cleaning:
review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64


In [ ]:
# Verify: check some real comments are still intact (not overwritten)
print("Sample of comments that had real text:")
print(reviews[reviews['review_comment_message'] != 'No comment']['review_comment_message'].head())

Sample of comments that had real text:
3                 Recebi bem antes do prazo estipulado.
4     Parabéns lojas lannister adorei comprar pela I...
9     aparelho eficiente. no site a marca do aparelh...
12      Mas um pouco ,travando...pelo valor ta Boa.\r\n
15    Vendedor confiável, produto ok e entrega antes...
Name: review_comment_message, dtype: object


In [ ]:
# Explore the orders dataset
print("Shape (rows, columns):", orders.shape)
print()
print("Missing values per column:")
print(orders.isnull().sum())
print()
print("First rows:")
orders.head()

Shape (rows, columns): (99441, 8)

Missing values per column:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

First rows:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [ ]:
# See the different order statuses
print("Order status counts:")
print(orders['order_status'].value_counts())

Order status counts:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [7]:
# Keep only delivered orders (the only ones with a real delivery time to analyze)
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print("Orders before filtering:", orders.shape[0])
print("Orders after keeping only 'delivered':", orders_delivered.shape[0])
print()
print("Missing delivery dates now:")
print(orders_delivered['order_delivered_customer_date'].isnull().sum())

Orders before filtering: 99441
Orders after keeping only 'delivered': 96478

Missing delivery dates now:
8


In [8]:
# Remove the 8 delivered orders that are missing a delivery date (data inconsistency)
orders_delivered = orders_delivered.dropna(subset=['order_delivered_customer_date']).copy()

print("Orders after removing rows with no delivery date:", orders_delivered.shape[0])
print("Missing delivery dates now:", orders_delivered['order_delivered_customer_date'].isnull().sum())

Orders after removing rows with no delivery date: 96470
Missing delivery dates now: 0


In [9]:
# Remove the 8 delivered orders that are missing a delivery date (data inconsistency)
orders_delivered = orders_delivered.dropna(subset=['order_delivered_customer_date']).copy()

print("Orders after removing rows with no delivery date:", orders_delivered.shape[0])
print("Missing delivery dates now:", orders_delivered['order_delivered_customer_date'].isnull().sum())

Orders after removing rows with no delivery date: 96470
Missing delivery dates now: 0


In [10]:
# Convert the date columns from text to real datetime format
orders_delivered['order_purchase_timestamp'] = pd.to_datetime(orders_delivered['order_purchase_timestamp'])
orders_delivered['order_delivered_customer_date'] = pd.to_datetime(orders_delivered['order_delivered_customer_date'])
orders_delivered['order_estimated_delivery_date'] = pd.to_datetime(orders_delivered['order_estimated_delivery_date'])

# Check the data types changed
print(orders_delivered[['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']].dtypes)

order_purchase_timestamp         datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [11]:
# Create delivery metrics (feature engineering)

# 1. Actual delivery time: days from purchase to delivery
orders_delivered['delivery_days'] = (orders_delivered['order_delivered_customer_date'] - orders_delivered['order_purchase_timestamp']).dt.days

# 2. Delivery vs promise: days early (negative) or late (positive) vs the estimated date
orders_delivered['delivery_vs_estimated'] = (orders_delivered['order_delivered_customer_date'] - orders_delivered['order_estimated_delivery_date']).dt.days

# Check the new columns
print(orders_delivered[['delivery_days', 'delivery_vs_estimated']].describe())

       delivery_days  delivery_vs_estimated
count   96470.000000           96470.000000
mean       12.093604             -11.875889
std         9.551380              10.182105
min         0.000000            -147.000000
25%         6.000000             -17.000000
50%        10.000000             -12.000000
75%        15.000000              -7.000000
max       209.000000             188.000000


In [12]:
# Merge orders (with delivery metrics) and reviews on order_id
merged = pd.merge(
    orders_delivered[['order_id', 'delivery_days', 'delivery_vs_estimated']],
    reviews[['order_id', 'review_score']],
    on='order_id',
    how='inner'
)

print("Merged table shape:", merged.shape)
print()
merged.head()

Merged table shape: (96353, 4)



,order_id,delivery_days,delivery_vs_estimated,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,8,-8,4
1,53cdb2fc8bc7dce0b6741e2150273451,13,-6,4
2,47770eb9100c2d0c44946d9cf07ec65d,9,-18,5
3,949d5b44dbf5de918fe9c16f97b45f8a,13,-13,5
4,ad21c59c0840e6cb83a9ceb5573f8159,2,-10,5


In [13]:
# The key question: do slower deliveries get worse reviews?
# Group by review score and see the average delivery metrics for each
analysis = merged.groupby('review_score')[['delivery_days', 'delivery_vs_estimated']].mean().round(1)

print("Average delivery metrics by review score:")
print(analysis)

Average delivery metrics by review score:
              delivery_days  delivery_vs_estimated
review_score                                      
1                      20.8                   -4.0
2                      16.2                   -8.6
3                      13.8                  -10.8
4                      11.8                  -12.4
5                      10.2                  -13.4


In [14]:
# Explore the order items dataset
print("Shape:", items.shape)
print()
print("Missing values:")
print(items.isnull().sum())
print()
items.head()

Shape: (112650, 7)

Missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64



,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [15]:
# Each order can have multiple items, so summarize to one row per order
order_values = items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum')
).reset_index()

print("Shape after grouping to one row per order:", order_values.shape)
print()
order_values.head()

Shape after grouping to one row per order: (98666, 3)



,order_id,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14


In [16]:
# Merge the order values (price, freight) into our main analysis table
merged = pd.merge(merged, order_values, on='order_id', how='inner')

print("Merged table shape:", merged.shape)
print()
merged.head()

Merged table shape: (96353, 6)



,order_id,delivery_days,delivery_vs_estimated,review_score,total_price,total_freight
0,e481f51cbdc54678b7cc49136f2d6af7,8,-8,4,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,13,-6,4,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,9,-18,5,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,13,-13,5,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2,-10,5,19.90,8.72


In [17]:
# Does price affect satisfaction? Group by review score and see average price and freight
price_analysis = merged.groupby('review_score')[['total_price', 'total_freight']].mean().round(2)

print("Average price and freight by review score:")
print(price_analysis)

Average price and freight by review score:
              total_price  total_freight
review_score                            
1                  164.91          28.16
2                  143.66          26.33
3                  127.48          23.55
4                  132.11          22.35
5                  134.43          21.71


In [18]:
# Explore the products dataset
print("Shape:", products.shape)
print()
print("Missing values:")
print(products.isnull().sum())
print()
products.head()

Shape: (32951, 9)

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64



,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [19]:
# Explore the category translation table
translation = pd.read_csv('product_category_name_translation.csv')
print("Shape:", translation.shape)
print()
translation.head()

Shape: (71, 2)



,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [20]:
# Step 1: Add English category names to the products table
products_en = pd.merge(products[['product_id', 'product_category_name']],
                       translation,
                       on='product_category_name',
                       how='left')

# Step 2: Connect products to orders through the items table
items_category = pd.merge(items[['order_id', 'product_id']],
                          products_en[['product_id', 'product_category_name_english']],
                          on='product_id',
                          how='left')

# Keep one category per order (the first product's category)
order_category = items_category.groupby('order_id')['product_category_name_english'].first().reset_index()

print("Shape:", order_category.shape)
print()
order_category.head()

Shape: (98666, 2)



,order_id,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools


In [21]:
# Merge category into the main analysis table
merged = pd.merge(merged, order_category, on='order_id', how='left')

# Analyze: average review score by product category (top and bottom categories)
category_analysis = merged.groupby('product_category_name_english')['review_score'].agg(['mean', 'count']).round(2)

# Only keep categories with enough orders to be meaningful (at least 100)
category_analysis = category_analysis[category_analysis['count'] >= 100]

# Sort to see the best and worst
category_analysis = category_analysis.sort_values('mean', ascending=False)

print("BEST rated categories:")
print(category_analysis.head(10))
print()
print("WORST rated categories:")
print(category_analysis.tail(10))

BEST rated categories:
                               mean  count
product_category_name_english             
books_general_interest         4.54    488
food_drink                     4.45    220
books_technical                4.43    256
luggage_accessories            4.38   1004
food                           4.32    431
stationery                     4.30   2242
fashion_shoes                  4.28    232
pet_shop                       4.28   1677
costruction_tools_garden       4.28    186
small_appliances               4.27    605

WORST rated categories:
                               mean  count
product_category_name_english             
telephony                      4.06   4056
home_construction              4.06    465
construction_tools_safety      4.02    153
bed_bath_table                 4.01   9191
fashion_underwear_beach        4.01    116
fixed_telephony                3.97    210
home_confort                   3.91    371
audio                          3.84    342
fashio

In [22]:
# First, create delivery time buckets (fast, medium, slow) for a cleaner pivot
merged['delivery_speed'] = pd.cut(merged['delivery_days'],
                                   bins=[0, 7, 15, 300],
                                   labels=['Fast (0-7 days)', 'Medium (8-15 days)', 'Slow (16+ days)'])

# Pivot table: average review score by delivery speed
pivot_speed = pd.pivot_table(merged,
                              values='review_score',
                              index='delivery_speed',
                              aggfunc='mean').round(2)

print("Pivot table - Average review score by delivery speed:")
print(pivot_speed)

Pivot table - Average review score by delivery speed:
                    review_score
delivery_speed                  
Fast (0-7 days)             4.41
Medium (8-15 days)          4.28
Slow (16+ days)             3.57


/tmp/ipykernel_2156/2033004390.py:7: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_speed = pd.pivot_table(merged,


In [23]:
# Create freight buckets (cheap vs expensive shipping)
merged['freight_level'] = pd.cut(merged['total_freight'],
                                  bins=[0, 15, 30, 1000],
                                  labels=['Cheap (0-15)', 'Medium (15-30)', 'Expensive (30+)'])

# Pivot table crossing TWO dimensions: delivery speed x freight level
pivot_cross = pd.pivot_table(merged,
                             values='review_score',
                             index='delivery_speed',
                             columns='freight_level',
                             aggfunc='mean',
                             observed=False).round(2)

print("Pivot table - Review score by delivery speed AND freight level:")
print(pivot_cross)

Pivot table - Review score by delivery speed AND freight level:
freight_level       Cheap (0-15)  Medium (15-30)  Expensive (30+)
delivery_speed                                                   
Fast (0-7 days)             4.46            4.42             4.10
Medium (8-15 days)          4.24            4.36             4.07
Slow (16+ days)             3.36            3.62             3.53


In [24]:
# Export the final analysis table for Tableau
merged.to_csv('olist_analysis_for_tableau.csv', index=False)
print("File exported: olist_analysis_for_tableau.csv")
print("Shape:", merged.shape)
print("Columns:", list(merged.columns))

File exported: olist_analysis_for_tableau.csv
Shape: (96353, 9)
Columns: ['order_id', 'delivery_days', 'delivery_vs_estimated', 'review_score', 'total_price', 'total_freight', 'product_category_name_english', 'delivery_speed', 'freight_level']


In [25]:
from google.colab import files
files.download('olist_analysis_for_tableau.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusions

This analysis explored which factors drive customer satisfaction (review scores)
in the Olist Brazilian e-commerce dataset, cleaning and combining several tables
covering ~96,000 delivered orders.

### Key Findings

**1. Delivery time is the strongest driver of satisfaction.**
Average delivery time drops steadily as review scores rise, from 20.8 days (1 star)
to 10.2 days (5 stars). Review scores collapse once delivery passes ~15 days
(from 4.28 average at 8-15 days down to 3.57 at 16+ days).

**2. Beating the delivery estimate matters, not just raw speed.**
5-star orders arrived ~13 days ahead of the promised date, versus only ~4 days
early for 1-star orders. Exceeding customer expectations drives higher ratings.

**3. Shipping cost has a clear effect; product price does not.**
Higher freight costs are associated with worse reviews (28.16 avg for 1 star vs
21.71 for 5 stars). Product price showed no consistent relationship.

**4. Product category affects satisfaction.**
Simple, easy-to-ship items rate highest (books 4.54, food & drink 4.45), while
large or complex items rate lowest (office furniture 3.65, audio 3.84).

### Recommendation
Olist should prioritize delivery speed above all, keeping deliveries under 15 days
and continuing to beat delivery estimates. Quality and logistics improvements
should focus on low-rated, high-volume categories like office furniture and
bed_bath_table.